# RetailPulse - Customer Behavior Analytics & Revenue Intelligence Platform

## Day 4 - Feature Engineering

### Objectives
- RFM score computation (Recency, Frequency, Monetary)
- Date-derived features (day-of-week, seasonality)
- Customer lifetime value approximation
- Save engineered customer-level feature table


In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("../data/processed/retail_cleaned.csv", parse_dates=["InvoiceDate"])
print("Shape:", df.shape)
df.head()

Shape: (805549, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 1. Date-derived features (transaction level)

In [2]:
df["InvoiceYear"] = df["InvoiceDate"].dt.year
df["InvoiceMonth"] = df["InvoiceDate"].dt.month
df["InvoiceWeekday"] = df["InvoiceDate"].dt.day_name()
df["InvoiceHour"] = df["InvoiceDate"].dt.hour
df["IsWeekend"] = df["InvoiceDate"].dt.dayofweek.isin([5, 6]).astype(int)
df["Quarter"] = df["InvoiceDate"].dt.quarter

df[["InvoiceDate", "InvoiceYear", "InvoiceMonth", "InvoiceWeekday", "InvoiceHour", "IsWeekend", "Quarter"]].head()

,InvoiceDate,InvoiceYear,InvoiceMonth,InvoiceWeekday,InvoiceHour,IsWeekend,Quarter
0,2009-12-01 07:45:00,2009,12,Tuesday,7,0,4
1,2009-12-01 07:45:00,2009,12,Tuesday,7,0,4
2,2009-12-01 07:45:00,2009,12,Tuesday,7,0,4
3,2009-12-01 07:45:00,2009,12,Tuesday,7,0,4
4,2009-12-01 07:45:00,2009,12,Tuesday,7,0,4


## 2. RFM Score Computation

- **Recency:** days since each customer's most recent purchase, measured from one day after the dataset's last transaction (the analysis "snapshot" date).
- **Frequency:** number of distinct invoices (orders) per customer.
- **Monetary:** total revenue per customer.


In [3]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalPrice", "sum")
).reset_index()

rfm.describe()

Snapshot date: 2011-12-10 12:50:00


,Customer ID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2745.861771
std,1715.572666,209.338707,13.009406,12211.648615
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,338.237500
50%,15314.500000,96.000000,3.000000,865.185000
75%,16797.750000,380.000000,7.000000,2238.807500
max,18287.000000,739.000000,398.000000,474728.370000


### RFM quintile scoring (1-5, 5 = best)

In [4]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)
rfm["RFM_Total"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,12346,326,12,580.86,2,5,2,252,9
1,12347,2,8,5591.72,5,4,5,545,14
2,12348,75,5,1821.40,3,4,4,344,11
3,12349,19,4,3642.94,5,3,5,535,13
4,12350,310,1,312.40,2,1,2,212,5


## 3. Customer Lifetime Value (approximation)

Simple CLV approximation: average order value x purchase frequency x estimated customer lifespan (in years, based on their observed tenure in the dataset). This is a heuristic CLV, not a probabilistic model (e.g. BG/NBD) - appropriate for a 10-day sprint, documented as such.

In [5]:
customer_tenure = df.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
customer_tenure["TenureDays"] = (customer_tenure["max"] - customer_tenure["min"]).dt.days
customer_tenure["TenureDays"] = customer_tenure["TenureDays"].replace(0, 1)  # avoid div-by-zero for single-purchase customers

rfm = rfm.merge(customer_tenure[["TenureDays"]], left_on="Customer ID", right_index=True)

rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["PurchaseFrequencyPerYear"] = rfm["Frequency"] / (rfm["TenureDays"] / 365.25).clip(lower=1/365.25)
rfm["CLV_Approx"] = rfm["AvgOrderValue"] * rfm["PurchaseFrequencyPerYear"] * (rfm["TenureDays"] / 365.25).clip(lower=1/365.25)

rfm[["Customer ID", "AvgOrderValue", "PurchaseFrequencyPerYear", "TenureDays", "CLV_Approx"]].describe()

,Customer ID,AvgOrderValue,PurchaseFrequencyPerYear,TenureDays,CLV_Approx
count,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,347.949605,120.244845,273.311160,2745.861771
std,1715.572666,325.050572,180.829332,258.503197,12211.648615
min,12346.000000,2.950000,1.023109,1.000000,2.950000
25%,13833.250000,176.760539,4.800659,1.000000,338.237500
50%,15314.500000,278.272500,9.967030,220.500000,865.185000
75%,16797.750000,409.775625,365.250000,511.000000,2238.807500
max,18287.000000,5816.220000,2191.500000,738.000000,474728.370000


## 4. Customer-level behavioral features (for churn model in Day 6)

In [6]:
customer_features = df.groupby("Customer ID").agg(
    UniqueProducts=("StockCode", "nunique"),
    TotalItemsPurchased=("Quantity", "sum"),
    AvgItemsPerOrder=("Quantity", "mean"),
    WeekendOrderRatio=("IsWeekend", "mean"),
    PrimaryCountry=("Country", lambda x: x.mode().iloc[0]),
).reset_index()

rfm_final = rfm.merge(customer_features, on="Customer ID")
rfm_final.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total,TenureDays,AvgOrderValue,PurchaseFrequencyPerYear,CLV_Approx,UniqueProducts,TotalItemsPurchased,AvgItemsPerOrder,WeekendOrderRatio,PrimaryCountry
0,12346,326,12,580.86,2,5,2,252,9,400,48.405,10.957500,580.86,27,270,7.941176,0.000000,United Kingdom
1,12347,2,8,5591.72,5,4,5,545,14,402,698.965,7.268657,5591.72,126,3246,12.830040,0.158103,Iceland
2,12348,75,5,1821.40,3,4,4,344,11,362,364.280,5.044890,1821.40,25,2714,53.215686,0.058824,Finland
3,12349,19,4,3642.94,5,3,5,535,13,570,910.735,2.563158,3642.94,138,1624,9.280000,0.000000,Italy
4,12350,310,1,312.40,2,1,2,212,5,1,312.400,365.250000,312.40,17,197,11.588235,0.000000,Norway


## 5. Save engineered feature table

In [7]:
rfm_final.to_csv("../data/processed/customer_features.csv", index=False)
print(f"Saved customer_features.csv - {rfm_final.shape[0]} customers, {rfm_final.shape[1]} features")
rfm_final.columns.tolist()

Saved customer_features.csv - 5878 customers, 18 features


['Customer ID',
 'Recency',
 'Frequency',
 'Monetary',
 'R_Score',
 'F_Score',
 'M_Score',
 'RFM_Score',
 'RFM_Total',
 'TenureDays',
 'AvgOrderValue',
 'PurchaseFrequencyPerYear',
 'CLV_Approx',
 'UniqueProducts',
 'TotalItemsPurchased',
 'AvgItemsPerOrder',
 'WeekendOrderRatio',
 'PrimaryCountry']

## Summary
- RFM computed for all customers using a snapshot date one day after the dataset's last transaction.
- CLV approximation is heuristic (avg order value x annualized frequency x tenure), not a probabilistic lifetime model - acceptable for sprint scope, flagged for the report.
- Output feeds directly into Day 5 (K-Means segmentation on Recency/Frequency/Monetary) and Day 6 (churn classifier, using the fuller behavioral feature set).

**Next:** `Day5_Segmentation.ipynb`
